# Cortex AMD GPU Setup (Pinggy SSH Tunnels)
Run these cells from top to bottom to host both your **Fast LLM** and **Embedding Model** using vLLM on your AMD GPUs.

This version uses `Pinggy`, which routes through the native SSH client built into your Jupyter notebook (no downloads or installations required!).

In [ ]:
import subprocess
import sys
import time

print(f"Using Python interpreter: {sys.executable}")

# 1. Start the Fast LLM Server in the background
# Using 70% of GPU Memory on Port 8000
llm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-7B-Instruct",
    "--port", "8000",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.7"
]

print("Starting Fast LLM Server...")
llm_process = subprocess.Popen(llm_cmd, stdout=open('llm.log', 'w'), stderr=subprocess.STDOUT)

# 2. Start the Embedding Model Server in the background
# Using 20% of GPU Memory on Port 8001
embed_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", "BAAI/bge-base-en-v1.5",
    "--port", "8001",
    "--max-model-len", "512",
    "--gpu-memory-utilization", "0.2"
]

print("Starting Embedding Server...")
embed_process = subprocess.Popen(embed_cmd, stdout=open('embed.log', 'w'), stderr=subprocess.STDOUT)

print("\nServers are booting up in the background!")
print("Waiting 15 seconds for models to load into VRAM before starting tunnels...")
time.sleep(15)
print("Models should be up! You can check llm.log and embed.log if you encounter issues.")

In [ ]:
# 3. Launch Pinggy SSH Tunnels and Print Config
import subprocess
import time
import os
import re

def start_pinggy_tunnel(port, name):
    print(f"Starting {name} tunnel via Pinggy on port {port}...")
    
    cmd = f"ssh -p 443 -o StrictHostKeyChecking=no -R0:localhost:{port} a.pinggy.io"
    process = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    
    os.set_blocking(process.stdout.fileno(), False)
    url = ""
    output = ""
    
    # Poll for 25 seconds
    for _ in range(25):
        try:
            chunk = process.stdout.read()
            if chunk:
                output += chunk
            
            urls = re.findall(r"https://[^\s]*pinggy[^\s]*", output)
            for u in urls:
                if "dashboard" not in u:
                    url = u
                    break
                    
            if url:
                print(f"✅ {name} Tunnel Active: {url}")
                break
        except Exception:
            pass
        time.sleep(1)
        
    if not url:
        print(f"⚠️ {name} tunnel started, but couldn't parse URL.")
        print(f"Raw SSH Output: {output}")
        
    return process, url

llm_proc, llm_url = start_pinggy_tunnel(8000, "Fast LLM")
embed_proc, embed_url = start_pinggy_tunnel(8001, "Embedding Model")

print("\n=======================================================")
print("PASTE THIS INTO YOUR LOCAL CORTEX backend/.env FILE:")
print("=======================================================")
if llm_url:
    print(f'FAST_MODEL_BASE_URL="{llm_url}/v1"')
if embed_url:
    print(f'EMBEDDING_MODEL_ENDPOINT="{embed_url}/v1"')
print("=======================================================")
print("(Pinggy tunnels don't have security warning screens!)")